# LatticeQCD.jl v2 入門（Colab・Jupyter）

[LatticeQCD.jl](https://github.com/akio-tomiya/LatticeQCD.jl) は、Julia で書かれた格子QCDのパッケージです。このノートブックでは、v2 の書き方（`SimulationSpec` → `build_simulation` → `run!`）で計算を走らせ、結果をグラフにします。

- Google Colab で開くと、ランタイムは自動で **Julia** になります（なっていなければ「ランタイム」→「ランタイムのタイプを変更」で Julia を選んでください）。
- 対応する Julia は 1.11 と 1.12 です。
- 動画（日本語）：準備中

## 1. 準備

LatticeQCD.jl と、グラフを描く PlotlyLight.jl を入れます。PlotlyLight は軽く、グラフを拡大したり、点に重ねて値を読んだりできます。

Colab では、このリポジトリの release に置いた事前コンパイル済みのファイルを取ってきて展開します。普通に入れると、Colab の CPU では事前コンパイルに 8 分ほどかかるためです。Colab の中身が変わっていて使えないときは、普通に入れます（`tools/colab_setup.jl`）。手元の Jupyter などでは、普通に `Pkg.add` します。

In [ ]:
using Pkg
if isdir("/content")   # Google Colab：事前コンパイル済みのファイルを使う / use the precompiled cache
    include(download("https://raw.githubusercontent.com/akio-tomiya/JuliaQCD-tutorials/plotlylight-test/tools/colab_setup.jl"))
else
    Pkg.add(["LatticeQCD", "PlotlyLight"])
end
VERSION

In [ ]:
using LatticeQCD, PlotlyLight, Statistics
pkgversion(LatticeQCD)

## 2. 最初の計算：2 フレーバーの Wilson フェルミオンで HMC

ノートブックでは対話式の `run_wizard()` が使いにくいので、設定を TOML で書いて読み込みます。中身は `run_wizard()` の simple モードの既定値とほぼ同じです（4⁴ の格子、β = 5.7、κ = 0.141139、cold start）。

In [ ]:
write("hmc.toml", """
["Physical setting"]
L = [4, 4, 4, 4]
"β" = 5.7
update_method = "HMC"
Nwing = 1
Nthermalization = 1
Nsteps = 51

["Physical setting(fermions)"]
Dirac_operator = "Wilson"
quench = false
hop = 0.141139

["HMC related"]
"Δτ" = 0.05
MDsteps = 20

["System Control"]
log_dir = "./logs"
logfile = "hmc.txt"
measurement_basedir = "./measurements"
measurement_dir = "hmc"
hasgradientflow = false

["Measurement set".measurement_methods.Plaquette]
methodname = "Plaquette"
measure_every = 1
""")

spec = load_simulation_spec("hmc.toml")

`RecordingSimulationEventSink` を渡しておくと、計算中の出来事（測った値を含む）が全てたまり、あとで Julia の中で使えます。`verbose=false` で、1 本ごとの表示を止めます。

In [ ]:
rec = RecordingSimulationEventSink()
session = build_simulation(spec, GaugefieldsEnvironment(); sink=rec)
@time result = run!(session; verbose=false)   # summary という名前は Julia の組み込み関数と重なるので使わない

In [ ]:
plaq = [(e.record.point.trajectory, e.record.value) for e in rec.events
        if e isa MeasurementFinished && e.record.name == :plaquette]
accepted = count(e -> e isa TrajectoryFinished && e.update.accepted, rec.events)
println("accepted: ", accepted, " / ", length(plaq))
p = plot.scatter(x=first.(plaq), y=last.(plaq), mode="lines+markers")
p.layout.xaxis.title.text = "trajectory"
p.layout.yaxis.title.text = "plaquette"
p

cold start ではプラケットは 1 から始まり、熱化すると 0.58 のあたりに落ち着きます。結果はファイルにも保存されます（`measurements/` と `logs/`）。

In [ ]:
readdir("measurements/hmc"), readdir("logs")

## 3. パイ中間子の質量を測る（クエンチ近似）

配位はグルーオンだけで作り（熱浴法と過緩和法）、パイ中間子の相関関数を Wilson フェルミオン（κ = 0.150）で測ります。Colab の CPU で数分で終わるように、格子は 4³×16 と小さくし、クォークも重め（κ = 0.150）にしています。動画では 8³×16、κ = 0.164 で測っています。

In [ ]:
write("pion.toml", """
["Physical setting"]
L = [4, 4, 4, 16]
"β" = 5.7
update_method = "Heatbath"
useOR = true
numOR = 3
Nthermalization = 21
Nsteps = 141

["Physical setting(fermions)"]
Dirac_operator = "nothing"

["System Control"]
log_dir = "./logs"
logfile = "pion.txt"
measurement_basedir = "./measurements"
measurement_dir = "pion"
hasgradientflow = false

["Measurement set".measurement_methods.Pion_correlator]
methodname = "Pion_correlator"
measure_every = 4

    ["Measurement set".measurement_methods.Pion_correlator.fermion_parameters]
    Dirac_operator = "Wilson"
    hop = 0.150
""")

spec_pion = load_simulation_spec("pion.toml")
rec_pion = RecordingSimulationEventSink()
session_pion = build_simulation(spec_pion, GaugefieldsEnvironment(); sink=rec_pion)
@time run!(session_pion; verbose=false)

In [ ]:
C = [e.record.value for e in rec_pion.events
     if e isa MeasurementFinished && e.record.name == :pion_correlator]
Cm = mean(C)
println("number of measurements: ", length(C))
p = plot.scatter(x=0:length(Cm)-1, y=log.(Cm), mode="lines+markers")
p.layout.xaxis.title.text = "t"
p.layout.yaxis.title.text = "log C(t)"
p

相関関数は、周期的な格子では $C(t) \propto \cosh(m_\pi (t - N_t/2))$ の形になります。隣り合う 3 つの時刻から有効質量 $m_{\rm eff}(t) = \cosh^{-1}\frac{C(t-1)+C(t+1)}{2C(t)}$ を求め、誤差はジャックナイフ法（1 回ずつ抜いて計算し直す）で見積もります。

In [ ]:
Nt = length(Cm)
meff(c) = [acosh(max(1, (c[t-1] + c[t+1]) / 2c[t])) for t in 2:Nt÷2+1]
n = length(C); S = sum(C)
jk = [meff((S - c) / (n - 1)) for c in C]
m = mean(jk); err = sqrt.((n - 1) / n * sum(x -> (x - m) .^ 2, jk))
p = plot.scatter(x=1:Nt÷2, y=m, mode="markers", error_y=(type="data", array=err))
p.layout.xaxis.title.text = "t"
p.layout.yaxis.title.text = "m_eff a"
p.layout.yaxis.range = [0, 2.5]
p

t が小さいところには重い状態が混ざり、t が大きくなると有効質量は一定の値（パイ中間子の質量）に近づきます。ここでは空間が 4 と小さく、測定も 30 回なので、平らな部分（t = 4〜8 で 0.85〜0.95 ほど）は揺れています。格子を大きくし（例えば 8³×16）、測定を増やすと、はっきりします。

## 次に

- 公式の README：https://github.com/akio-tomiya/LatticeQCD.jl
- v1 から v2 への移行：`docs/src/migration-v2.md`
- 質問：[JuliaQCD のディスカッション](https://github.com/orgs/JuliaQCD/discussions/7)